# Cheat Sheet — Time Series Diagnostics (ACF, PACF, White Noise, Stationarity, CCF)

Quick-reference code snippets for exploratory time-series diagnostics. Organized by "how do I...?" — copy/paste and adapt. Minimal narrative by design; see `01_Background_Theory...md` for the underlying theory.


## Setup (imports you'll almost always need)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose
from scipy.signal import correlate


## Load a CSV time series with a datetime index

In [ ]:
data = pd.read_csv('yourfile.csv', header=0, index_col=0)
data.index = pd.to_datetime(data.index, format='%Y-%m')   # adjust format to match your dates
data.plot(figsize=(10, 4))


## Plot ACF / PACF

In [ ]:
plot_acf(series, lags=50, alpha=0.05)     # shaded band = ~95% CI under white-noise null
plot_pacf(series, lags=50, alpha=0.05)    # PACF -> candidate AR order; ACF shape -> candidate MA order
plt.show()


## First-order (and higher-order) differencing

In [ ]:
diffed_once = np.diff(series, n=1)          # removes a linear trend; numpy array, one shorter
diffed_twice = np.diff(series, n=2)         # rarely needed; watch for over-differencing (neg. spike at lag 1)

# pandas equivalent, keeps the index:
diffed_pd = series.diff(1).dropna()


## Seasonal differencing (for a known period, e.g. 12 = monthly data with yearly cycle)

In [ ]:
seasonal_diffed = series.diff(12).dropna()
# combine with a first difference if trend AND seasonality are both present:
seasonal_then_first = series.diff(12).diff(1).dropna()


## Log transform (stabilize variance that grows with the level / multiplicative seasonality)

In [ ]:
log_series = np.log(series)          # series must be strictly positive
# common combo: log -> seasonal diff -> first diff
transformed = np.log(series).diff(12).diff(1).dropna()


## White noise: generate & visually/statistically check

In [ ]:
white_noise = np.random.normal(loc=0, scale=1, size=1000)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(white_noise); ax[0].axhline(0, color='r')
plot_acf(white_noise, ax=ax[1])
plt.show()

acorr_ljungbox(white_noise, lags=[10, 30, 50], return_df=True)
# H0: no autocorrelation. p < 0.05 -> reject H0 -> NOT white noise (autocorrelation present)


## Ljung-Box test on model residuals

In [ ]:
acorr_ljungbox(model_residuals, lags=[10, 20], return_df=True)
# Apply this to residuals after fitting ANY model (regression, ARIMA, ...) to check
# whether the model has captured all the serial correlation.


## ADF and KPSS stationarity tests (use together)

In [ ]:
def stationarity_report(series, name="series"):
    s = pd.Series(series).dropna()
    adf_stat, adf_p, *_ = adfuller(s, autolag='AIC')
    kpss_stat, kpss_p, *_ = kpss(s, regression='c', nlags='auto')
    print(f"--- {name} ---")
    print(f"ADF : p={adf_p:.4f}  -> {'stationary' if adf_p < 0.05 else 'NON-stationary'}  (H0: unit root / non-stationary)")
    print(f"KPSS: p={kpss_p:.4f}  -> {'NON-stationary' if kpss_p < 0.05 else 'stationary'}  (H0: stationary)")

stationarity_report(series, "raw")


**ADF vs. KPSS quick-reference table**

| ADF | KPSS | Read as |
|---|---|---|
| stationary | stationary | Strong evidence of stationarity |
| non-stationary | non-stationary | Strong evidence of non-stationarity — difference it |
| stationary | non-stationary | Possibly trend-stationary — try detrending, not differencing |
| non-stationary | stationary | Ambiguous / borderline — inspect visually, try one difference |


## Seasonal decomposition (trend / seasonal / residual)

In [ ]:
decomp = seasonal_decompose(series, model='additive')   # or model='multiplicative'
decomp.plot()
plt.show()
# access components directly:
decomp.trend, decomp.seasonal, decomp.resid


## Rolling statistics (quick manual stationarity check)

In [ ]:
roll_mean = series.rolling(window=12).mean()
roll_std = series.rolling(window=12).std()

plt.plot(series, label='original')
plt.plot(roll_mean, label='rolling mean')
plt.plot(roll_std, label='rolling std')
plt.legend(); plt.show()
# Rolling mean/std that visibly drift over time -> non-stationary


## Boxplot by year/period (visual variance-and-level check)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
sns.boxplot(x=data.index.year, y=data.iloc[:, 0], ax=ax)
ax.set(xlabel='Year', ylabel=data.columns[0])


## Cross-correlation function (CCF) between two series

In [ ]:
def plot_ccf(data_a, data_b, lag_lookback, percentile=95):
    n = len(data_a)
    ccf = correlate(data_a - np.mean(data_a), data_b - np.mean(data_b), method='direct') / (
        np.std(data_a) * np.std(data_b) * n
    )
    _min = (len(ccf) - 1) // 2 - lag_lookback
    _max = (len(ccf) - 1) // 2 + (lag_lookback - 1)
    z = {90: 1.645, 95: 1.96, 99: 2.576}[percentile] / np.sqrt(n)

    plt.figure(figsize=(12, 4))
    plt.stem(np.arange(-lag_lookback, lag_lookback - 1), ccf[_min:_max])
    plt.axhline(z, color='b', ls='--'); plt.axhline(-z, color='b', ls='--')
    plt.axvline(0, color='black')
    plt.title('Cross-Correlation'); plt.xlabel('Lag'); plt.ylabel('Correlation')
    plt.show()
    return ccf

# Interpretation: peak at lag k > 0 means data_a leads data_b by k steps
# (data_a's past is correlated with data_b's present).
plot_ccf(series_a, series_b, lag_lookback=50)


## Aligning a leading indicator once you know the lag

In [ ]:
best_lag = 2   # found from the CCF peak
aligned_leading_series = series_a.shift(best_lag)
# now regress / correlate `series_b` against `aligned_leading_series`


## Rule-of-thumb table: reading ACF/PACF shapes → candidate model

| ACF shape | PACF shape | Candidate model |
|---|---|---|
| Cuts off after lag q | Tails off / decays | MA(q) |
| Tails off / decays | Cuts off after lag p | AR(p) |
| Tails off | Tails off | ARMA(p, q) — both present |
| Slow, near-linear decay | — | Non-stationary — difference first |
| Spikes repeating every s lags | — | Seasonal component of period s — seasonal differencing |

General guidance: avoid AR/MA orders much above ~5 for a single-variable model; high orders on real data are usually overfitting a short, noisy sample rather than genuine structure.


## Common pitfalls checklist

- **Forgot to difference before computing ACF/CCF** → a trend dominates and hides the real (short-lag) correlation structure.
- **Over-differenced** → an artificial strong negative spike appears at lag 1; try one fewer difference.
- **Used only ADF or only KPSS** → their null hypotheses are opposite; use both.
- **Applied a difference when the series only needed a log transform** (variance grows with level, but no real trend in the mean) → detrend/transform, don't over-difference.
- **Ran Ljung-Box at only one lag** → false positives/negatives happen; check a small range of lags (e.g., 10, 20, 50).
- **Ignored seasonality** → strong periodic ACF spikes remain after a first difference; add a seasonal difference.
- **Treated a single ADF/KPSS run as final** → always pair with a visual check (raw plot, rolling mean/std, decomposition).
